## Bussiness understanding

Our business requirement is to find semantically similar documents based on user-provided sentences.

## Data Collection

### Import necessary libraries

In [1]:
import os
import re
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from gensim.models import FastText
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

In [7]:
documents = []

DATASET_PATH = Path(r"R:\Learning\Core_NLP\02_Word2Vec\Projects\Document_similarity-engine\News Articles")

# Read all TXT files from category folders

for file_path in DATASET_PATH.rglob("*.txt"):

    # Extract document ID from filename
    document_id = file_path.stem

    # Extract category from parent folder
    category = file_path.parent.name

    # Read article content
    with open(file_path, "r", encoding="utf-8", errors="replace") as file:
        article = file.read()

    # Store collected information
    documents.append({
        "document_id": document_id,
        "category": category,
        "article": article,
        "document_key": f"{category}_{document_id}"
    })

# Convert collected data into DataFrame
document_data = pd.DataFrame(documents)

print("\nData collection completed!")


Data collection completed!


In [8]:
document_data

,document_id,category,article,document_key
0,001,business,Ad sales boost Time Warner profit\n\nQuarterly...,business_001
1,002,business,Dollar gains on Greenspan speech\n\nThe dollar...,business_002
2,003,business,Yukos unit buyer faces loan claim\n\nThe owner...,business_003
3,004,business,High fuel prices hit BA's profits\n\nBritish A...,business_004
4,005,business,Pernod takeover talk lifts Domecq\n\nShares in...,business_005
...,...,...,...,...
95,016,tech,Global blogger action day called\n\nThe global...,tech_016
96,017,tech,Finding new homes for old phones\n\nRe-using o...,tech_017
97,018,tech,PlayStation 3 chip to be unveiled\n\nDetails o...,tech_018
98,019,tech,Intel unveils laser breakthrough\n\nIntel has ...,tech_019


In [9]:
print("\nFirst 5 records:")
print(document_data.head())

print("\nTotal documents collected:")
print(len(document_data))

print("\nCategory names:")
print(document_data["category"].unique())


First 5 records:
  document_id  category                                            article  \
0         001  business  Ad sales boost Time Warner profit\n\nQuarterly...   
1         002  business  Dollar gains on Greenspan speech\n\nThe dollar...   
2         003  business  Yukos unit buyer faces loan claim\n\nThe owner...   
3         004  business  High fuel prices hit BA's profits\n\nBritish A...   
4         005  business  Pernod takeover talk lifts Domecq\n\nShares in...   

   document_key  
0  business_001  
1  business_002  
2  business_003  
3  business_004  
4  business_005  

Total documents collected:
100

Category names:
<StringArray>
['business', 'entertainment', 'politics', 'sport', 'tech']
Length: 5, dtype: str


## Data Understanding

### Exploratory Data Analysis

In [11]:
# 1. Total number of documents
print("\nTotal Documents:")
print(len(document_data))


# 2. Category-wise document distribution
print("\nCategory Distribution:")
print(document_data["category"].value_counts())


# 3. Identify empty documents
empty_documents = document_data[
    document_data["article"].fillna("").str.strip() == ""]

print("\nEmpty Documents:")
print(len(empty_documents))


# 4. Identify duplicate documents
duplicate_documents = document_data[
    document_data["article"].fillna("").str.strip().duplicated(keep=False)
    & (document_data["article"].fillna("").str.strip() != "")
]

print("\nDuplicate Documents:")
print(len(duplicate_documents))


# Display duplicate records
print("\nDuplicate Document Details:")
print(duplicate_documents[["document_id", "category"]])



Total Documents:
100

Category Distribution:
category
business         20
entertainment    20
politics         20
sport            20
tech             20
Name: count, dtype: int64

Empty Documents:
0

Duplicate Documents:
2

Duplicate Document Details:
   document_id category
71         012    sport
79         020    sport


## Data Preparation

In [13]:
# Remove the Duplicate document

document_data = document_data[
    document_data["article"].fillna("").str.strip() != ""
].copy()

# Reset the index values
document_data.reset_index(drop=True, inplace=True)

### Data Cleaning

In [14]:
stop_words = set(stopwords.words("english"))
stop_words = stop_words - {"no", "not", "never"}

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Sentence tokenization
    sentences = sent_tokenize(text)

    processed_sentences = []

    for sentence in sentences:

        # Remove numbers and unwanted characters
        sentence = re.sub(r"[^a-z\s]", " ", sentence)

        # Remove extra whitespace
        sentence = re.sub(r"\s+", " ", sentence).strip()

        # Word tokenization
        words = word_tokenize(sentence)

        # Stopword removal
        words = [
            word for word in words
            if word not in stop_words and len(word) > 1
        ]

        # Keep sentences containing meaningful words
        if words:
            processed_sentences.append(words)

    return processed_sentences


document_data["processed_sentences"] = document_data["article"].apply(preprocess_text)

print("\nPreprocessing completed!")


Preprocessing completed!


In [15]:
sample = "Artificial Intelligence is NOT replacing human. Do you think it will change the humans ?"

result = preprocess_text(sample)

print(result)

[['artificial', 'intelligence', 'not', 'replacing', 'human'], ['think', 'change', 'humans']]


## Model Building & Training

In [16]:
training_corpus = [
    sentence
    for document_sentences in document_data["processed_sentences"]
    for sentence in document_sentences
]

model = FastText(
    sentences=training_corpus,
    vector_size=50,
    window=3,
    min_count=2,
    sg=0,
    min_n=3,
    max_n=6,
    workers=4,
    seed=42,
    epochs=20
)


print("\nFastText model created & Trained successfully!")


FastText model created & Trained successfully!


## Model Training

## Document vector generation

In [17]:
def get_document_vector(processed_sentences, model):

    # Combine all words from the document
    words = [
        word
        for sentence in processed_sentences
        for word in sentence
    ]

    # Get FastText word vectors
    word_vectors = [
        model.wv[word]
        for word in words
    ]

    # Handle empty documents
    if not word_vectors:
        return None

    # Calculate average word embedding
    document_vector = np.mean(word_vectors, axis=0)

    return document_vector


# Generate document vectors
document_data["document_vector"] = document_data["processed_sentences"].apply(
    lambda sentences: get_document_vector(sentences, model)
)


# Keep documents having valid vectors
valid_documents = document_data[
    document_data["document_vector"].notna()
].copy()


# Reset index
valid_documents.reset_index(drop=True, inplace=True)


# Create document vector DataFrame
document_vectors_df = valid_documents[
    ["document_key", "document_id", "category", "document_vector"]
].copy()

valid_documents

,document_id,category,article,document_key,processed_sentences,document_vector
0,001,business,Ad sales boost Time Warner profit\n\nQuarterly...,business_001,"[[ad, sales, boost, time, warner, profit, quar...","[0.372836, 0.428223, -0.34172, 0.184225, 1.053..."
1,002,business,Dollar gains on Greenspan speech\n\nThe dollar...,business_002,"[[dollar, gains, greenspan, speech, dollar, hi...","[0.372236, 0.426564, -0.341664, 0.184067, 1.04..."
2,003,business,Yukos unit buyer faces loan claim\n\nThe owner...,business_003,"[[yukos, unit, buyer, faces, loan, claim, owne...","[0.339066, 0.388919, -0.311496, 0.166333, 0.95..."
3,004,business,High fuel prices hit BA's profits\n\nBritish A...,business_004,"[[high, fuel, prices, hit, ba, profits, britis...","[0.369366, 0.422557, -0.338712, 0.183182, 1.04..."
4,005,business,Pernod takeover talk lifts Domecq\n\nShares in...,business_005,"[[pernod, takeover, talk, lifts, domecq, share...","[0.323856, 0.373502, -0.297541, 0.161475, 0.91..."
...,...,...,...,...,...,...
95,016,tech,Global blogger action day called\n\nThe global...,tech_016,"[[global, blogger, action, day, called, global...","[0.382962, 0.438104, -0.351831, 0.187701, 1.08..."
96,017,tech,Finding new homes for old phones\n\nRe-using o...,tech_017,"[[finding, new, homes, old, phones, using, old...","[0.40198, 0.464487, -0.369107, 0.198732, 1.138..."
97,018,tech,PlayStation 3 chip to be unveiled\n\nDetails o...,tech_018,"[[playstation, chip, unveiled, details, chip, ...","[0.381773, 0.438336, -0.35053, 0.188778, 1.077..."
98,019,tech,Intel unveils laser breakthrough\n\nIntel has ...,tech_019,"[[intel, unveils, laser, breakthrough, intel, ...","[0.372801, 0.428488, -0.342506, 0.182587, 1.05..."


## Model Testing


In [19]:
# Convert document vectors into NumPy matrix
document_matrix = np.vstack(
    document_vectors_df["document_vector"].values
)


def search_documents(input_text, top_n=5):

    # Step 1: Preprocess input sentence
    processed_input = preprocess_text(input_text)

    # Step 2: Generate input vector
    input_vector = get_document_vector(processed_input, model)

    if input_vector is None:
        print("\nNo usable word vectors found in input.")
        return pd.DataFrame()

    # Step 3: Calculate cosine similarity
    scores = cosine_similarity(
        input_vector.reshape(1, -1),
        document_matrix
    )[0]

    # Step 4: Create results DataFrame
    results = document_vectors_df[
        ["document_id", "category", "document_key"]
    ].copy()

    results["similarity_score"] = scores

    # Step 5: Sort by similarity score
    results = results.sort_values(
        by="similarity_score",
        ascending=False
    )

    # Step 6: Return Top N documents
    results = results.head(top_n).reset_index(drop=True)

    return results

In [22]:
search_documents("slicon")

,document_id,category,document_key,similarity_score
0,015,business,business_015,0.999969
1,019,tech,tech_019,0.999966
2,010,tech,tech_010,0.999965
3,014,tech,tech_014,0.999964
4,006,tech,tech_006,0.999964
